# Automotive Industry Trends Forecasting using Neural Networks

> **Supervised and Unsupervised Learning Course | HdM Stuttgart**  
> Authors: John Torres, Samuel Hempelt

[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/john2408/neural_networks_project_hdm)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)


##  Notebook (2/2): Multivariate Time Series with Vectorized Batching

This notebook demonstrates **multivariate time series forecasting** for German vehicle registration trends using a **vectorized batching approach** with exogenous features. We forecast monthly registration volumes across 1,502 individual series (OEM × Model × Powertrain combinations) using data from January 2018 to September 2025.

### Modeling Approach

Our forecasting pipeline follows a rigorous **three-fold cross-temporal validation** strategy to ensure model robustness across different time periods:

![Train-Test-Val Process](./img/Train_Test_Val_process.png)

Each fold trains on historical data and tests on a distinct future period:
- **Fold 1**: Train until Sep 2024 → Test Oct-Dec 2024
- **Fold 2**: Train until Dec 2024 → Test Jan-Mar 2025
- **Fold 3**: Train until Jun 2025 → Test Jul-Sep 2025

This walk-forward validation mimics real-world deployment where models are retrained as new data becomes available.

## Why Vectorized Batching with Exogenous Features?

Traditional approaches create one sample per (series, time_window) pair:
- 1,502 series × 83 windows = **124,666 individual samples**
- ~1,948 forward passes per epoch (batch_size=64)
- Low GPU utilization (5-15%)

```
Paradigm: ONE sample = ALL series (vectorized) + ONE time window
Identity: Implicit (position in series dimension)
Batching: Sequential time windows
```

The vectorized approach creates one sample per time window containing ALL series:
- 83 time windows (each containing 1,502 series)
- 83 training samples total
- ~6 forward passes per epoch (batch_size=16)
- High GPU utilization (80-95%)
- 20-50x faster training with identical results


## Dataset Configuration

Given **1,502 series × 93 timesteps**, with `seq_length=6`, `embargo=1`, and `test_period=9 months`:

```python
n_windows_total = n_timesteps - seq_length - embargo - test_period
                = 93 - 6 - 1 - 9
                = 77 total windows

# After 80/20 train-val split:
n_windows_train = int(77 * 0.8) = 61 windows
n_windows_val   = 77 - 61        = 16 windows

# Training samples:
dataset_length_train = 61 training samples
dataset_length_val   = 16 validation samples
```

**Total predictions per epoch**: 61 windows × 1,502 series = **91,622 forecasts**


## Sample Structure (TimeSeriesDatasetVectorizedExog)

Each sample represents **ALL series at ONE time window**:

```python
X.shape = (n_series, seq_length, n_features)
        = (1502, 6, 25)

# Features (25 total):
# - 1 target value (registration volume)
# - 24 exogenous features (GDP, CPI, unemployment, etc.)

y.shape = (1502,)

# Example window #45 (all series):
[
  [[val_0, gdp_0, cpi_0, …], ..., [val_5, gdp_5, cpi_5, …]],  # Series 0: t-5 to t
  [[val_0, gdp_0, cpi_0, …], ..., [val_5, gdp_5, cpi_5, …]],  # Series 1: t-5 to t
  ...
  [[val_0, gdp_0, cpi_0, …], ..., [val_5, gdp_5, cpi_5, …]]   # Series 1502: t-5 to t
]
# Shape: (1502, 6, 25)

# Target y (all series at t+1+embargo):
[1302.4, 2433.9, ..., 10096.0]
# Shape: (1502,)
```


## Batch Structure

With `batch_size=16`:

```python
X_batch.shape = (batch_time, n_series, seq_length, n_features)
              = (16, 1502, 6, 25)

y_batch.shape = (batch_time, n_series)
              = (16, 1502)

# Before feeding to model, reshape:
X_reshaped = X_batch.view(16*1502, 6, 25)  # (24032, 6, 25)
y_reshaped = y_batch.view(16*1502)         # (24032,)
```

**Total predictions per batch**: 16 windows × 1,502 series = **24,032 forecasts**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
GLOBAL_PATH = "/content/drive/MyDrive/HDM/"

In [ ]:
# log into wandb account and paste the API key when prompted
import wandb
wandb.login()

In [ ]:

import pandas as pd

path = "/content/drive/MyDrive/HDM/data/gold/monthly_registration_volume_gold.parquet"
df = pd.read_parquet(path)

In [ ]:
df.head()

In [ ]:
!pip install git+https://github.com/john2408/neural_networks_project_hdm.git

In [ ]:
MODEL = 'BASELINE'  # Options: 'LSTM', 'RNN', 'GRU', 'CNN1D', 'MLP', 'Transformer', 'BASELINE', 'NBEATS', 'NBEATSx'
EXOG = False  # Use exogenous features
ENTITY_NAME = "tensor-torres"  
PROJECT_NAME = "neuralnetworks-timeseries-multivariate" 

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import warnings
import time
import os
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import wandb

from neuralts.core.models import (LSTMForecaster, RNNForecaster, GRUForecaster, 
                            CNN1DForecaster, MLPForecaster, TransformerForecaster, TransformerForecasterCLS)
from neuralts.core.metrics import smape, calculate_smape_distribution
from neuralts.core.func import (TimeSeriesDatasetVectorizedExog, 
                    generate_out_of_sample_predictions_vectorized_exog, WandBLoggingCallback)

from neuralforecast.losses.pytorch import MSE, MAE


warnings.filterwarnings('ignore')


if __name__ == "__main__":


    # ================s========================================================
    # TRAINING PARAMETERS
    # ========================================================================
    

    PYTORCH_SEED = 42
    LOSS = MAE() # criterion = nn.L1Loss()
    criterion = nn.L1Loss() # equivalent to MAE

    # HYPERPARAMETER OPTIMIZATION WITH OPTUNA
    OPTIMIZE_HYPERPARAMETERS = True  # Set to False to skip optimization
    N_TRIALS = 5  # Number of Optuna trials
    OPTUNA_TIMEOUT = 900  # Timeout in seconds 15 minutes

    SEQ_LENGTH = 6
    TRAIN_RATIO = 0.8
    EMBARGO = 1
    EPOCHS = 30
    BATCH_SIZE = 8
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-5

    # -----------------------------------------------------------------------

    MLP_LAYERS = 2
    MLP_HIDDEN_SIZE = 512
    MLP_DROPOUT = 0.2

    LSTM_LAYERS = 2
    LSTM_HIDDEN_SIZE = 128
    LSTM_DROPOUT = 0.2

    RNN_LAYERS = 2
    RNN_HIDDEN_SIZE = 128
    RNN_DROPOUT = 0.2

    GRU_LAYERS = 2
    GRU_HIDDEN_SIZE = 128
    GRU_DROPOUT = 0.2

    CNN_LAYERS = 3
    CNN_HIDDEN_SIZE = 64
    CNN_DROPOUT = 0.2

    TRANSFORMER_D_MODEL = 64
    TRANSFORMER_NHEAD = 4
    TRANSFORMER_LAYERS = 2
    TRANSFORMER_DIM_FEEDFORWARD = 256
    TRANSFORMER_DROPOUT = 0.2
    
    # NBEATSx hyperparameters
    NBEATSX_DROPOUT = 0.5
    NBEATSX_MAX_STEPS = 500
    NBEATSX_N_HARMONICS = 2
    NBEATSX_N_BASIS = 2
    NBEATSX_N_BLOCKS = [1, 1, 1]
    NBEATSX_MLP_UNITS = [[512, 512], [512, 512], [512, 512]]

    # NBEATS hyperparameters
    NBEATS_DROPOUT = 0.5
    NBEATS_MAX_STEPS = 500
    NBEATS_N_HARMONICS = 2
    NBEATS_N_BASIS = 2
    NBEATS_N_BLOCKS = [1, 1, 1]
    NBEATS_MLP_UNITS = [[512, 512], [512, 512], [512, 512]]

    TOTAL_SCRIPT_RUNTIME = None
    TOTAL_TRAINING_TIME_FOLDS = dict()
    TOTAL_OPTIMIZATION_TIME = None



    GLOBAL_PATH = os.getcwd()

    MODE = None
    if not EXOG:
        MODE = "MULTI_VEC"
        MODEL_FOLDER = MODEL.lower() + "_" + MODE.lower()
    else:
        MODE = "MULTI_VEC_EXOG"
        MODEL_FOLDER = MODEL.lower() + "_" + MODE.lower()
        
    OUTPUT_PATH = os.path.join(GLOBAL_PATH, "models", MODEL_FOLDER)
    os.makedirs(OUTPUT_PATH, exist_ok=True)
    df_path = os.path.join(GLOBAL_PATH, "data", "gold", "monthly_registration_volume_gold_padding.parquet")

    # ========================================================================

    torch.manual_seed(PYTORCH_SEED)
    np.random.seed(PYTORCH_SEED)
    torch.cuda.manual_seed_all(PYTORCH_SEED)
    torch.backends.cudnn.deterministic = True # Ensures reproducibility
    torch.backends.cudnn.benchmark = False # Ensures reproducibility

    SCRIPT_START_TIME = time.time()

    # Check MPS availability
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print(f"✓ MPS device is available")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"✓ CUDA device is available")
    else:
        device = torch.device("cpu")
        print(f"Using CPU")
        
    print(f"Device: {device}")


    df_full = pd.read_parquet(df_path, engine='pyarrow')

    cols_to_drop = ["OEM", "Model", "drive_type"]
    df_full = df_full.drop(columns=cols_to_drop)
    
    # Keep only required columns for vectorized dataset
    base_columns = ['Date', 'ts_key', 'Value']
    exog_columns = [df for df in df_full.columns if df not in base_columns]
    
    if not EXOG:
        exog_columns = []

    df_full = df_full[['Date', 'ts_key', 'Value'] + exog_columns].copy()
    
    # Validate not NaN or infinite values
    assert not df_full['Value'].isna().any(), "NaN values found in Value column"
    assert not np.isinf(df_full['Value']).any(), "Infinite values found in Value column"

    print("="*80)
    print("DATA OVERVIEW")
    print("="*80)
    print(f"Original data shape: {df_full.shape}")
    print(f"Unique time series: {df_full.ts_key.nunique()}")
    print(f"Date range: {df_full.Date.min()} to {df_full.Date.max()}")
    print(f"Total observations: {len(df_full):,}")
    

    # Check original data
    print(f"\nOriginal DataFrame:")
    print(f"  NaN values: {df_full.isna().sum().sum()}")
    print(f"  Inf values: {np.isinf(df_full.select_dtypes(include=[np.number])).sum().sum()}")
    print(f"  Value range: [{df_full['Value'].min():.2f}, {df_full['Value'].max():.2f}]")
    
    # Check for zero/very small values that could cause division issues
    print(f"\nZero values in 'Value' column: {(df_full['Value'] == 0).sum()}")
    print(f"Values < 0.01: {(df_full['Value'] < 0.01).sum()}")
    
    # ========================================================================
    # FOLD CONFIGURATION - As per Problem Description
    # ========================================================================
    
    folds = [
        {
            'name': 'Fold 1',
            'train_end': '2024-09-30',    # Train on data up to Sep 2024
            'test_start': '2024-10-01',   # Test on Oct-Dec 2024
            'test_end': '2024-12-31'
        },
        {
            'name': 'Fold 2',
            'train_end': '2024-12-31',    # Train on data up to Dec 2024
            'test_start': '2025-01-01',   # Test on Jan-Mar 2025
            'test_end': '2025-03-31'
        },
        {
            'name': 'Fold 3',
            'train_end': '2025-06-30',    # Train on data up to Jun 2025
            'test_start': '2025-07-01',   # Test on Jul-Sep 2025
            'test_end': '2025-09-30'
        }
    ]
    
    print("\n" + "="*80)
    print("FOLD CONFIGURATION")
    print("="*80)
    for fold in folds:
        print(f"\n{fold['name']}:")
        print(f"  Training data: up to {fold['train_end']}")
        print(f"  Test period: {fold['test_start']} to {fold['test_end']}")
    
    
    # ========================================================================
    # HYPERPARAMETER OPTIMIZATION WITH OPTUNA
    # ========================================================================
    best_trial = None
    if OPTIMIZE_HYPERPARAMETERS and MODEL not in ['BASELINE']:
        import optuna
        from neuralts.core.func import create_model_from_trial, train_with_early_stopping_vectorized
        
        print("\n" + "="*80)
        print("HYPERPARAMETER OPTIMIZATION WITH OPTUNA")
        print("="*80)
        print(f"Model: {MODEL}")
        print(f"Trials: {N_TRIALS}")
        print(f"Timeout: {OPTUNA_TIMEOUT}s")
        
        # Use Fold 1 data for hyperparameter optimization
        fold_config = folds[0]
        df_train = df_full[df_full['Date'] <= fold_config['train_end']].copy()
        
        print(f"\nUsing {fold_config['name']} for optimization")
        print(f"  Training observations: {len(df_train):,}")
        
        # Create vectorized datasets with exogenous features
        train_dataset = TimeSeriesDatasetVectorizedExog(
            df_train,
            exog_cols=exog_columns,
            seq_length=SEQ_LENGTH,
            embargo=EMBARGO,
            train=True,
            train_ratio=TRAIN_RATIO
        )
        
        val_dataset = TimeSeriesDatasetVectorizedExog(
            df_train,
            exog_cols=exog_columns,
            seq_length=SEQ_LENGTH,
            train=False,
            embargo=EMBARGO,
            train_ratio=TRAIN_RATIO,
            scaler_X=train_dataset.scaler_X,
            scaler_y=train_dataset.scaler_y
        )
        
        # Input size: features per timestep (no one-hot encoding needed)
        INPUT_SIZE_OPT = train_dataset.X.shape[3]  # Shape: (n_windows, n_series, seq_length, n_features)
        
        # Store original dataframe for NeuralForecast models
        train_dataset_df = df_train.copy()
        
        print(f"\nOptimization dataset:")
        print(f"  Training samples: {len(train_dataset):,}")
        print(f"  Validation samples: {len(val_dataset):,}")
        print(f"  Input size: {INPUT_SIZE_OPT}")
        
        def objective(trial):
            """
            Optuna objective function for hyperparameter optimization with vectorized batching.
            """
            # Suggest hyperparameters
            learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
            batch_size = trial.suggest_categorical('batch_size', [4, 8, 16])  # Smaller batches for vectorized
            
            # Create model with trial hyperparameters
            model, hyperparams = create_model_from_trial(
                MODEL, trial, INPUT_SIZE_OPT, SEQ_LENGTH
            )
            
            # Special handling for NeuralForecast models (NBEATS, NBEATSx)
            if MODEL in ['NBEATS', 'NBEATSx']:
                from neuralforecast import NeuralForecast
                from neuralforecast.models import NBEATS, NBEATSx
                wb_callback = MetricsLoggingCallback()  # No W&B run during optimization
                
                # Prepare data in NeuralForecast format
                df_train_nf = train_dataset_df.copy()
                df_train_nf = df_train_nf.rename(columns={'ts_key': 'unique_id', 'Date': 'ds', 'Value': 'y'})
                
                # Split into train/val based on chronological split
                n_unique_dates = len(df_train_nf['ds'].unique())
                val_size = int(n_unique_dates * (1 - TRAIN_RATIO))
                
                if MODEL == 'NBEATS':
                    df_train_nf = df_train_nf[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds'])
                    
                    nf_model = NBEATS(
                        input_size=SEQ_LENGTH,
                        loss=LOSS,
                        h=val_size,
                        #dropout_prob_theta=hyperparams['dropout_prob_theta'],
                        max_steps=hyperparams['max_steps'],
                        n_harmonics=hyperparams['n_harmonics'],
                        n_polynomials=hyperparams['n_polynomials'],
                        n_blocks=hyperparams['n_blocks'],
                        mlp_units=hyperparams['mlp_units'],
                        scaler_type='robust',
                        random_seed=PYTORCH_SEED, 
                        callbacks=[wb_callback] 
                    )
                    
                    nf = NeuralForecast(models=[nf_model], freq='ME')
                    nf.fit(df=df_train_nf, val_size=val_size)

                                        # Get validation loss from trainer callback metrics
                    if hasattr(nf.models[0], 'metrics'):
                        best_val_loss = nf.models[0].metrics.get('valid_loss', float('inf'))
                    else:
                        raise RuntimeError("Trainer or callback_metrics not found in NeuralForecast model.")
                        #best_val_loss = float('inf')
                    
                    return best_val_loss
                    
                elif MODEL == 'NBEATSx':
                    assert EXOG, "Exogenous features must be enabled for NBEATSx model."
                    df_train_nf = df_train_nf[['unique_id', 'ds', 'y'] + exog_columns].sort_values(['unique_id', 'ds'])

                    # Create static dataframe with one-hot encoded series IDs
                    unique_ids = df_train_nf['unique_id'].unique()
                    static_df = pd.DataFrame({'unique_id': unique_ids})
                    for uid in unique_ids:
                        static_df[f'id_{uid}'] = (static_df['unique_id'] == uid).astype(int)
                    
                    stat_exog_list = [f'id_{uid}' for uid in unique_ids]
                    
                    nf_model = NBEATSx(
                        input_size=SEQ_LENGTH,
                        h=val_size,
                        loss=LOSS,
                        dropout_prob_theta=hyperparams['dropout_prob_theta'],
                        max_steps=hyperparams['max_steps'],
                        n_harmonics=hyperparams['n_harmonics'],
                        n_polynomials=hyperparams['n_polynomials'],
                        n_blocks=hyperparams['n_blocks'],
                        mlp_units=hyperparams['mlp_units'],
                        scaler_type='robust',
                        stat_exog_list=stat_exog_list,
                        futr_exog_list=exog_columns,
                        random_seed=PYTORCH_SEED, 
                        callbacks=[wb_callback] 
                    )
                    
                    nf = NeuralForecast(models=[nf_model], freq='ME')
                    nf.fit(df=df_train_nf, static_df=static_df, val_size=val_size)
                
                # Get validation loss from trainer callback metrics
                if hasattr(nf.models[0], 'metrics'):
                    best_val_loss = nf.models[0].metrics.get('valid_loss', float('inf'))
                else:
                    raise RuntimeError("Trainer or callback_metrics not found in NeuralForecast model.")
                    #best_val_loss = float('inf')
                
                return best_val_loss
            
            else:
                # Standard PyTorch models
                model = model.to(device)
                
                # Create data loaders with vectorized dataset
                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
                val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
                
                # Train with vectorized early stopping
                best_val_loss, _ = train_with_early_stopping_vectorized(
                    model, train_loader, val_loader, device, SEQ_LENGTH,
                    learning_rate=learning_rate,
                    weight_decay=WEIGHT_DECAY,
                    max_epochs=EPOCHS,
                    patience=5
                )
                
                return best_val_loss
        
        # Create and run study
        study = optuna.create_study(
            direction='minimize',
            study_name=f'{MODEL}_optimization',
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
        )
        
        OPTIMIZATION_START_TIME = time.time()
        study.optimize(objective, n_trials=N_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=True)
        TOTAL_OPTIMIZATION_TIME = (time.time() - OPTIMIZATION_START_TIME) / 60.0  # in minutes
        print(f"\nOptimization completed in {TOTAL_OPTIMIZATION_TIME:.1f} minutes")

        print("\n" + "="*80)
        print("OPTIMIZATION RESULTS")
        print("="*80)
        print(f"Number of finished trials: {len(study.trials)}")
        print(f"\nBest trial:")
        best_trial = study.best_trial
        print(f"  Value (Validation Loss): {best_trial.value:.6f}")
        print(f"  Hyperparameters:")
        for key, value in best_trial.params.items():
            print(f"    {key}: {value}")
        
        # Update hyperparameters with best trial values
        BATCH_SIZE = best_trial.params.get('batch_size', BATCH_SIZE)
        LEARNING_RATE = best_trial.params.get('learning_rate', LEARNING_RATE)
        
        if MODEL == 'MLP':
            MLP_LAYERS = best_trial.params.get('num_layers', MLP_LAYERS)
            MLP_HIDDEN_SIZE = best_trial.params.get('hidden_size', MLP_HIDDEN_SIZE)
            MLP_DROPOUT = best_trial.params.get('dropout', MLP_DROPOUT)
        
        elif MODEL == 'LSTM':
            LSTM_LAYERS = best_trial.params.get('num_layers', LSTM_LAYERS)
            LSTM_HIDDEN_SIZE = best_trial.params.get('hidden_size', LSTM_HIDDEN_SIZE)
            LSTM_DROPOUT = best_trial.params.get('dropout', LSTM_DROPOUT)
        
        elif MODEL == 'RNN':
            RNN_LAYERS = best_trial.params.get('num_layers', RNN_LAYERS)
            RNN_HIDDEN_SIZE = best_trial.params.get('hidden_size', RNN_HIDDEN_SIZE)
            RNN_DROPOUT = best_trial.params.get('dropout', RNN_DROPOUT)
        
        elif MODEL == 'GRU':
            GRU_LAYERS = best_trial.params.get('num_layers', GRU_LAYERS)
            GRU_HIDDEN_SIZE = best_trial.params.get('hidden_size', GRU_HIDDEN_SIZE)
            GRU_DROPOUT = best_trial.params.get('dropout', GRU_DROPOUT)
        
        elif MODEL == 'CNN1D':
            CNN_LAYERS = best_trial.params.get('num_layers', CNN_LAYERS)
            CNN_HIDDEN_SIZE = best_trial.params.get('hidden_size', CNN_HIDDEN_SIZE)
            CNN_DROPOUT = best_trial.params.get('dropout', CNN_DROPOUT)
        
        elif MODEL in ['Transformer', 'TransformerCLS']:
            TRANSFORMER_D_MODEL = best_trial.params.get('d_model', TRANSFORMER_D_MODEL)
            TRANSFORMER_NHEAD = best_trial.params.get('nhead', TRANSFORMER_NHEAD)
            TRANSFORMER_LAYERS = best_trial.params.get('num_layers', TRANSFORMER_LAYERS)
            TRANSFORMER_DIM_FEEDFORWARD = best_trial.params.get('dim_feedforward', TRANSFORMER_DIM_FEEDFORWARD)
            TRANSFORMER_DROPOUT = best_trial.params.get('dropout', TRANSFORMER_DROPOUT)
        
        elif MODEL == 'NBEATS':
            NBEATS_DROPOUT = best_trial.params.get('dropout_prob_theta', NBEATS_DROPOUT)
            NBEATS_MAX_STEPS = best_trial.params.get('max_steps', NBEATS_MAX_STEPS)
            NBEATS_N_HARMONICS = best_trial.params.get('n_harmonics', NBEATS_N_HARMONICS)
            NBEATS_N_BASIS = best_trial.params.get('n_polynomials', NBEATS_N_BASIS)
            NBEATS_N_BLOCKS = best_trial.params.get('n_blocks', NBEATS_N_BLOCKS)
            mlp_size = best_trial.params.get('mlp_size', 512)
            NBEATS_MLP_UNITS = [[mlp_size, mlp_size]] * len(NBEATS_N_BLOCKS)
        
        elif MODEL == 'NBEATSx':
            NBEATSX_DROPOUT = best_trial.params.get('dropout_prob_theta', NBEATSX_DROPOUT)
            NBEATSX_MAX_STEPS = best_trial.params.get('max_steps', NBEATSX_MAX_STEPS)
            NBEATSX_N_HARMONICS = best_trial.params.get('n_harmonics', NBEATSX_N_HARMONICS)
            NBEATSX_N_BASIS = best_trial.params.get('n_polynomials', NBEATSX_N_BASIS)
            NBEATSX_N_BLOCKS = best_trial.params.get('n_blocks', NBEATSX_N_BLOCKS)
            mlp_size = best_trial.params.get('mlp_size', 512)
            NBEATSX_MLP_UNITS = [[mlp_size, mlp_size]] * len(NBEATSX_N_BLOCKS)
        
        print(f"\n✓ Hyperparameters updated with best trial values")
        if MODEL not in ['NBEATS', 'NBEATSx']:
            print(f"  Learning rate: {LEARNING_RATE:.6f}")
            print(f"  Batch size: {BATCH_SIZE}")
        
        # Save optimization results
        optuna_results_path = os.path.join(OUTPUT_PATH, 'optuna_results.csv')
        trials_df = study.trials_dataframe()
        trials_df.to_csv(optuna_results_path, index=False)
        print(f"\n✓ Saved Optuna results to: {optuna_results_path}")
        
        # Save best hyperparameters
        best_params_path = os.path.join(OUTPUT_PATH, 'best_hyperparameters.txt')
        with open(best_params_path, 'w') as f:
            f.write(f"Best Hyperparameters for {MODEL}\n")
            f.write("="*60 + "\n\n")
            f.write(f"Validation Loss: {best_trial.value:.6f}\n\n")
            f.write("Parameters:\n")
            for key, value in best_trial.params.items():
                f.write(f"  {key}: {value}\n")
        print(f"✓ Saved best hyperparameters to: {best_params_path}")
    
    elif MODEL == 'BASELINE':
        print(f"\n⚠ Skipping hyperparameter optimization for {MODEL} model")
    else:
        print("\n⚠ Hyperparameter optimization disabled (OPTIMIZE_HYPERPARAMETERS=False)")
    
    
    # ========================================================================
    # FOLD-WISE TRAINING AND EVALUATION
    # ========================================================================
    
    fold_metrics = []
    fold_smape_distributions = []
    import matplotlib.pyplot as plt
    
    for fold_idx, fold_config in enumerate(folds):
        
        # Fold Variables
        fold_output_dir = os.path.join(OUTPUT_PATH, f"fold_{fold_idx + 1}")
        os.makedirs(fold_output_dir, exist_ok=True)

        print("\n" + "="*80)
        print(f"{fold_config['name'].upper()}: {fold_config['test_start']} to {fold_config['test_end']}")
        print("="*80)
        
        # Initialize W&B for this fold
        wandb_run = None
        # Prepare config based on model type
        wandb_config = {
            "model": MODEL,
            "fold": fold_config['name'],
            "seq_length": SEQ_LENGTH,
            "embargo": EMBARGO,
            "train_ratio": TRAIN_RATIO,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "exogenous_features": EXOG,
            "n_exog_features": len(exog_columns) if EXOG else 0,
            "train_end": fold_config['train_end'],
            "test_start": fold_config['test_start'],
            "test_end": fold_config['test_end']
        }
        
        # Add model-specific hyperparameters
        if MODEL == 'MLP':
            wandb_config.update({
                "num_layers": MLP_LAYERS,
                "hidden_size": MLP_HIDDEN_SIZE,
                "dropout": MLP_DROPOUT
            })
        elif MODEL == 'LSTM':
            wandb_config.update({
                "num_layers": LSTM_LAYERS,
                "hidden_size": LSTM_HIDDEN_SIZE,
                "dropout": LSTM_DROPOUT
            })
        elif MODEL == 'RNN':
            wandb_config.update({
                "num_layers": RNN_LAYERS,
                "hidden_size": RNN_HIDDEN_SIZE,
                "dropout": RNN_DROPOUT
            })
        elif MODEL == 'GRU':
            wandb_config.update({
                "num_layers": GRU_LAYERS,
                "hidden_size": GRU_HIDDEN_SIZE,
                "dropout": GRU_DROPOUT
            })
        elif MODEL == 'CNN1D':
            wandb_config.update({
                "num_layers": CNN_LAYERS,
                "hidden_size": CNN_HIDDEN_SIZE,
                "dropout": CNN_DROPOUT
            })
        elif MODEL in ['Transformer', 'TransformerCLS']:
            wandb_config.update({
                "d_model": TRANSFORMER_D_MODEL,
                "nhead": TRANSFORMER_NHEAD,
                "num_layers": TRANSFORMER_LAYERS,
                "dim_feedforward": TRANSFORMER_DIM_FEEDFORWARD,
                "dropout": TRANSFORMER_DROPOUT
            })
        elif MODEL in ['NBEATS']:
            wandb_config.update({
                "dropout_prob_theta": NBEATS_DROPOUT,
                "max_steps": NBEATS_MAX_STEPS,
                "n_harmonics": NBEATS_N_HARMONICS,
                "n_basis": NBEATS_N_BASIS,
                "n_blocks": NBEATS_N_BLOCKS,
                "mlp_units": NBEATS_MLP_UNITS
            })
        elif MODEL in ['NBEATSx']:
            wandb_config.update({
                "dropout_prob_theta": NBEATSX_DROPOUT,
                "max_steps": NBEATSX_MAX_STEPS,
                "n_harmonics": NBEATSX_N_HARMONICS,
                "n_basis": NBEATSX_N_BASIS,
                "n_blocks": NBEATSX_N_BLOCKS,
                "mlp_units": NBEATSX_MLP_UNITS
            })
        
        # Initialize W&B run (for all models including BASELINE)
        if MODEL != 'BASELINE':
            wandb_run = wandb.init(
                entity=ENTITY_NAME,
                project=PROJECT_NAME,
                name=f"{MODEL}__{MODE}__{fold_config['name'].replace(' ', '_')}",
                config=wandb_config,
                reinit=True  # Allow multiple runs in same script
            )
            print(f"\n✓ Initialized W&B run: {wandb_run.name}")
        
        # Filter training data up to train_end date
        df_train = df_full[df_full['Date'] <= fold_config['train_end']].copy()
        
        print(f"\nFiltered training data up to {fold_config['train_end']}")
        print(f"  Training observations: {len(df_train):,}")
        print(f"  Date range: {df_train['Date'].min()} to {df_train['Date'].max()}")
        
        # -----------------------------------------------------------------
        # BASELINE MODEL: Moving Average
        # -----------------------------------------------------------------
        if MODEL == 'BASELINE':
            print("\n" + "-"*60)
            print("BASELINE MODEL: Calculating Moving Average Predictions")
            print("-"*60)
            
            # Get test period data
            df_test_period = df_full[
                (df_full['Date'] >= fold_config['test_start']) & 
                (df_full['Date'] <= fold_config['test_end'])
            ].copy()
            
            # Calculate moving average baseline for each time series
            predictions_dict = {}
            actuals_dict = {}
            all_preds = []
            all_acts = []
            
            unique_ts_keys = df_test_period['ts_key'].unique()
            print(f"Generating baseline predictions for {len(unique_ts_keys)} time series...")
            
            for ts_key in unique_ts_keys:
                # Get last SEQ_LENGTH values up to train_end for this time series
                ts_train_data = df_train[df_train['ts_key'] == ts_key].sort_values('Date')
                
                if len(ts_train_data) < SEQ_LENGTH + EMBARGO:
                    # If not enough history, use all available data
                    baseline_value = ts_train_data['Value'].mean()
                else:
                    # Use last SEQ_LENGTH values, skipping the EMBARGO period
                    baseline_value = ts_train_data['Value'].iloc[-(SEQ_LENGTH + EMBARGO):-EMBARGO].mean()

                # Get actual values for this time series in test period
                ts_test_data = df_test_period[df_test_period['ts_key'] == ts_key].sort_values('Date')
                actual_values = ts_test_data['Value'].values
                
                # Predict the same baseline value for all test periods
                predicted_values = np.full(len(actual_values), baseline_value)

                # Round prediction to 1 decimal place
                predicted_values = np.round(predicted_values, 1)
                
                # Replance NaN prediction with zero
                predicted_values = np.where(np.isnan(predicted_values), 0, predicted_values)

                predictions_dict[ts_key] = predicted_values
                actuals_dict[ts_key] = actual_values
                all_preds.extend(predicted_values)
                all_acts.extend(actual_values)
            
            all_preds = np.array(all_preds)
            all_acts = np.array(all_acts)
            
            print(f"✓ Generated {len(all_preds)} baseline predictions")
            
            # Initialize W&B for BASELINE model
            wandb_run = wandb.init(
                entity=ENTITY_NAME,
                project=PROJECT_NAME,
                name=f"{MODEL}__{MODE}__{fold_config['name'].replace(' ', '_')}",
                config=wandb_config,
                reinit=True
            )
            print(f"\n✓ Initialized W&B run: {wandb_run.name}")
        
        # -----------------------------------------------------------------
        # NBEATS MODEL: Neural Basis Expansion Analysis
        # -----------------------------------------------------------------
        elif MODEL == 'NBEATS':
            print("\n" + "-"*60)
            print("NBEATS MODEL: Neural Basis Expansion Analysis")
            print("-"*60)
            
            from neuralforecast import NeuralForecast
            from neuralforecast.models import NBEATS
            
            # Prepare data in neuralforecast format
            df_train_nf = df_train.copy()
            df_train_nf = df_train_nf.rename(columns={'ts_key': 'unique_id', 'Date': 'ds', 'Value': 'y'})
            df_train_nf = df_train_nf[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds'])
            
            # Get test period data
            df_test_period = df_full[
                (df_full['Date'] >= fold_config['test_start']) & 
                (df_full['Date'] <= fold_config['test_end'])
            ].copy()
            
            # Calculate horizon (number of months to forecast)
            horizon = len(df_test_period['Date'].unique())
            
            print(f"Training NBEATS with input_size={SEQ_LENGTH}, horizon={horizon}")
            
            wb_callback = WandBLoggingCallback(wandb_run)

            # Initialize model
            models = [
                NBEATS(
                    loss=LOSS,
                    input_size=SEQ_LENGTH,
                    h=horizon,
                    max_steps=NBEATS_MAX_STEPS,
                    scaler_type='robust',
                    random_seed=PYTORCH_SEED, 
                    n_harmonics=NBEATS_N_HARMONICS,
                    n_basis=NBEATS_N_BASIS,
                    n_blocks=NBEATS_N_BLOCKS,
                    mlp_units=NBEATS_MLP_UNITS,
                    #dropout_prob_theta=NBEATS_DROPOUT,
                    callbacks=[wb_callback] 
                )                
            ]

            
            nf = NeuralForecast(models=models, freq='ME')  # ME = Month End
        
            # Train model
            nf.fit(df=df_train_nf, val_size=horizon)
            
            # Generate predictions
            Y_hat_df = nf.predict().reset_index()
            

            # Prepare test data in same format
            df_test_nf = df_test_period.copy()
            df_test_nf = df_test_nf.rename(columns={'ts_key': 'unique_id', 'Date': 'ds', 'Value': 'y'})
            df_test_nf = df_test_nf[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds'])
            
            # Merge predictions with actuals
            df_results = pd.merge(df_test_nf, Y_hat_df, on=['unique_id', 'ds'], how='left')
            
            # Extract predictions and actuals per time series
            predictions_dict = {}
            actuals_dict = {}
            
            for ts_key in df_results['unique_id'].unique():
                ts_data = df_results[df_results['unique_id'] == ts_key].sort_values('ds')
                preds = ts_data['NBEATS'].values
                acts = ts_data['y'].values
                
                # Replace negative predictions with 0
                preds = np.maximum(preds, 0)
                
                # Handle NaN predictions
                preds = np.nan_to_num(preds, nan=0.0)
                
                predictions_dict[ts_key] = preds
                actuals_dict[ts_key] = acts
            
            all_preds = np.concatenate([v for v in predictions_dict.values()])
            all_acts = np.concatenate([v for v in actuals_dict.values()])
            
            print(f"✓ Generated {len(all_preds)} NBEATS predictions")
        
        # -----------------------------------------------------------------
        # NBEATSx MODEL: NBEATS with Exogenous Variables
        # -----------------------------------------------------------------
        elif MODEL == 'NBEATSx':
            print("\n" + "-"*60)
            print("NBEATSx MODEL: NBEATS with Exogenous Variables")
            print("-"*60)

            assert EXOG, "NBEATSx model requires EXOG=True to include exogenous features"
            
            from neuralforecast import NeuralForecast
            from neuralforecast.models import NBEATSx
            
            wb_callback = WandBLoggingCallback(wandb_run)
            
            # Prepare data in neuralforecast format
            df_train_nf = df_train.copy()
            df_train_nf = df_train_nf.rename(columns={'ts_key': 'unique_id', 'Date': 'ds', 'Value': 'y'})
            
         
            # Include exogenous features
            df_train_nf = df_train_nf[['unique_id', 'ds', 'y'] + exog_columns].sort_values(['unique_id', 'ds'])

            # Get test period data
            df_test_period = df_full[
                (df_full['Date'] >= fold_config['test_start']) & 
                (df_full['Date'] <= fold_config['test_end'])
            ].copy()
            
            # Calculate horizon (number of months to forecast)
            horizon = len(df_test_period['Date'].unique())
            
            # Create static dataframe with one-hot encoded series IDs
            unique_ids = df_train_nf['unique_id'].unique()
            static_df = pd.DataFrame({'unique_id': unique_ids})
            
            # One-hot encode the series IDs
            for uid in unique_ids:
                static_df[f'id_{uid}'] = (static_df['unique_id'] == uid).astype(int)
            
            print(f"Training NBEATSx with input_size={SEQ_LENGTH}, horizon={horizon}, exog={EXOG}")
            print(f"Static features (one-hot encoded series): {len(unique_ids)}")
            if EXOG:
                print(f"Future exogenous features: {exog_columns}")
            
            # Initialize model with stat_exog_list and optional futr_exog_list
            stat_exog_list = [f'id_{uid}' for uid in unique_ids]
            

            models = [
                NBEATSx(
                    loss=LOSS,
                    input_size=SEQ_LENGTH,
                    h=horizon,
                    dropout_prob_theta=NBEATSX_DROPOUT,
                    max_steps=NBEATSX_MAX_STEPS,
                    n_harmonics=NBEATSX_N_HARMONICS,
                    n_polynomials=NBEATSX_N_BASIS,
                    n_blocks=NBEATSX_N_BLOCKS,
                    mlp_units=NBEATSX_MLP_UNITS,
                    scaler_type='robust',
                    stat_exog_list=stat_exog_list,
                    futr_exog_list=exog_columns,
                    random_seed=PYTORCH_SEED,
                    callbacks=[wb_callback]
                )
            ]

            nf = NeuralForecast(models=models, freq='ME')  # ME = Month End
            
            # Train model
            print("Training NBEATSx model...")
            nf.fit(df=df_train_nf, static_df=static_df, val_size=horizon)
            
            # Log training metrics from NeuralForecast trainer
            if hasattr(nf.models[0], 'trainer') and hasattr(nf.models[0].trainer, 'callback_metrics'):
                metrics = nf.models[0].trainer.callback_metrics
                for metric_name, metric_value in metrics.items():
                    if isinstance(metric_value, torch.Tensor):
                        wandb_run.log({f"train_{metric_name}": metric_value.item()})
                print(f"✓ Logged {len(metrics)} training metrics to W&B")
            

            # Prepare future exogenous features for prediction
            df_test_nf = df_test_period.copy()
            df_test_nf = df_test_nf.rename(columns={'ts_key': 'unique_id', 'Date': 'ds', 'Value': 'y'})
            df_test_nf = df_test_nf[['unique_id', 'ds'] + exog_columns].sort_values(['unique_id', 'ds'])
            Y_hat_df = nf.predict(futr_df=df_test_nf).reset_index()

            
            # Prepare test data in same format
            df_test_nf_full = df_test_period.copy()
            df_test_nf_full = df_test_nf_full.rename(columns={'ts_key': 'unique_id', 'Date': 'ds', 'Value': 'y'})
            df_test_nf_full = df_test_nf_full[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds'])
            
            # Merge predictions with actuals
            df_results = pd.merge(df_test_nf_full, Y_hat_df, on=['unique_id', 'ds'], how='left')
            
            # Extract predictions and actuals per time series
            predictions_dict = {}
            actuals_dict = {}
            
            for ts_key in df_results['unique_id'].unique():
                ts_data = df_results[df_results['unique_id'] == ts_key].sort_values('ds')
                preds = ts_data['NBEATSx'].values
                acts = ts_data['y'].values
                
                # Replace negative predictions with 0
                preds = np.maximum(preds, 0)
                
                # Handle NaN predictions
                preds = np.nan_to_num(preds, nan=0.0)
                
                predictions_dict[ts_key] = preds
                actuals_dict[ts_key] = acts
            
            all_preds = np.concatenate([v for v in predictions_dict.values()])
            all_acts = np.concatenate([v for v in actuals_dict.values()])
            
            print(f"✓ Generated {len(all_preds)} NBEATSx predictions")
   
        else:
            # -----------------------------------------------------------------
            # STEP 1: Create VECTORIZED datasets with EXOGENOUS features for model development
            # -----------------------------------------------------------------
            print("\n" + "-"*60)
            print("STEP 1: Creating VECTORIZED datasets with EXOGENOUS features for model development")
            print("-"*60)
        
            train_dataset = TimeSeriesDatasetVectorizedExog(
                df_train,
                exog_cols=exog_columns,
                seq_length=SEQ_LENGTH,
                embargo=EMBARGO,
                train=True,
                train_ratio=TRAIN_RATIO
            )
            
            print(f"Training time windows: {len(train_dataset):,}")
            print(f"Series per window: {train_dataset.n_series:,}")
            print(f"Effective training samples: {len(train_dataset) * train_dataset.n_series:,}")
            
            # Save scalers and metadata
            scaler_X = train_dataset.scaler_X
            scaler_y = train_dataset.scaler_y
            n_series = train_dataset.n_series
            ts_key_to_idx = train_dataset.ts_key_to_idx
            
            test_dataset = TimeSeriesDatasetVectorizedExog(
                df_train,
                exog_cols=exog_columns,
                seq_length=SEQ_LENGTH,
                train=False,
                embargo=EMBARGO,
                train_ratio=TRAIN_RATIO,
                scaler_X=scaler_X,
                scaler_y=scaler_y
            )
            
            print(f"Validation time windows: {len(test_dataset):,}")
            print(f"Effective validation samples: {len(test_dataset) * train_dataset.n_series:,}")
        
        # -----------------------------------------------------------------
        # STEP 2: Initialize and train model with VECTORIZED batching
        # -----------------------------------------------------------------
            print("\n" + "-"*60)
            print(f"STEP 2: Training {MODEL} model with VECTORIZED batching")
            print("-"*60)
            
            # Input size: features per timestep (no one-hot encoding)
            INPUT_SIZE = train_dataset.X.shape[3]  # (n_windows, n_series, seq_length, n_features)
            print(f"Input size per timestep: {INPUT_SIZE} features")
            
            if MODEL == 'LSTM':
                model = LSTMForecaster(
                    input_size=INPUT_SIZE,
                    hidden_size=LSTM_HIDDEN_SIZE,
                    num_layers=LSTM_LAYERS,
                    dropout=LSTM_DROPOUT
                ).to(device)
            elif MODEL == 'RNN':
                model = RNNForecaster(
                    input_size=INPUT_SIZE,
                    hidden_size=RNN_HIDDEN_SIZE,
                    num_layers=RNN_LAYERS,
                    dropout=RNN_DROPOUT
                ).to(device)
            elif MODEL == 'GRU':
                model = GRUForecaster(
                    input_size=INPUT_SIZE,
                    hidden_size=GRU_HIDDEN_SIZE,
                    num_layers=GRU_LAYERS,
                    dropout=GRU_DROPOUT
                ).to(device)
            elif MODEL == 'CNN1D':
                model = CNN1DForecaster(
                    input_size=INPUT_SIZE,
                    hidden_size=CNN_HIDDEN_SIZE,
                    num_layers=CNN_LAYERS,
                    dropout=CNN_DROPOUT
                ).to(device)
            elif MODEL == 'MLP':
                model = MLPForecaster(
                    input_size=INPUT_SIZE,
                    seq_length=SEQ_LENGTH,
                    hidden_size=MLP_HIDDEN_SIZE,
                    num_layers=MLP_LAYERS,
                    dropout=MLP_DROPOUT
                ).to(device)
            elif MODEL == 'Transformer':
                model = TransformerForecaster(
                    input_size=INPUT_SIZE,
                    d_model=TRANSFORMER_D_MODEL,
                    nhead=TRANSFORMER_NHEAD,
                    num_layers=TRANSFORMER_LAYERS,
                    dim_feedforward=TRANSFORMER_DIM_FEEDFORWARD,
                    dropout=TRANSFORMER_DROPOUT
                ).to(device)
            elif MODEL == 'TransformerCLS':
                model = TransformerForecasterCLS(
                    input_size=INPUT_SIZE,
                    d_model=TRANSFORMER_D_MODEL,
                    nhead=TRANSFORMER_NHEAD,
                    num_layers=TRANSFORMER_LAYERS,
                    dim_feedforward=TRANSFORMER_DIM_FEEDFORWARD,
                    dropout=TRANSFORMER_DROPOUT
                ).to(device)
            else:
                raise ValueError(f"Unsupported MODEL type: {MODEL}")
        
            print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
            
            # Use smaller batch size for vectorized (each batch processes all series)
            VECTORIZED_BATCH_SIZE = 16
            train_loader = DataLoader(train_dataset, batch_size=VECTORIZED_BATCH_SIZE, shuffle=False)
            val_loader = DataLoader(test_dataset, batch_size=VECTORIZED_BATCH_SIZE, shuffle=False)
            
            print(f"Batch size: {VECTORIZED_BATCH_SIZE} time windows (processes {VECTORIZED_BATCH_SIZE * n_series:,} predictions per batch)")
            
    
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
            
            best_val_loss = float('inf')
            train_losses = []
            val_losses = []
            
            start_time = time.time()
            
            # Training loop with VECTORIZED batching
            for epoch in range(EPOCHS):
                # Training
                model.train()
                total_train_loss = 0
                
                for X_batch, y_batch in train_loader:
                    # X_batch: (batch_time, n_series, seq_length, n_features)
                    # Reshape to (batch_time * n_series, seq_length, n_features)
                    X_batch = X_batch.reshape(-1, SEQ_LENGTH, INPUT_SIZE).to(device)
                    y_batch = y_batch.reshape(-1, 1).to(device)
                    
                    optimizer.zero_grad()
                    predictions = model(X_batch)
                    loss = criterion(predictions, y_batch)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    
                    total_train_loss += loss.item()
                
                train_loss = total_train_loss / len(train_loader)
                
                # Validation
                model.eval()
                total_val_loss = 0
                
                with torch.no_grad():
                    for X_batch, y_batch in val_loader:
                        X_batch = X_batch.reshape(-1, SEQ_LENGTH, INPUT_SIZE).to(device)
                        y_batch = y_batch.reshape(-1, 1).to(device)
                        
                        predictions = model(X_batch)
                        loss = criterion(predictions, y_batch)
                        total_val_loss += loss.item()
                
                val_loss = total_val_loss / len(val_loader)
                
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                
                scheduler.step(val_loss)
                
                # Log metrics to W&B
                if wandb_run is not None:
                    wandb_run.log({
                        "epoch": epoch + 1,
                        "train_loss": train_loss,
                        "val_loss": val_loss,
                        "learning_rate": optimizer.param_groups[0]['lr']
                    })
                
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_model_state = model.state_dict().copy()
                
                if (epoch + 1) % 10 == 0 or epoch == 0:
                    elapsed = time.time() - start_time
                    print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | Train: {train_loss:.6f} | Val: {val_loss:.6f} | Time: {elapsed:.1f}s")
            
            # Load best model
            model.load_state_dict(best_model_state)

            training_time = time.time() - start_time  # in seconds

            TOTAL_TRAINING_TIME_FOLDS[fold_config['name']] = training_time
            
            print(f"\nTraining completed in {training_time:.1f} seconds")
            print(f"Best validation loss: {best_val_loss:.6f}")
            
            print(f"\nTraining completed in {time.time() - start_time:.1f}s")
            print(f"Best validation loss: {best_val_loss:.6f}")
            print(f"Vectorized speedup: Processed {n_series:,} series simultaneously per time window!")
            

            # -----------------------------------------------------------------
            # STEP 3: Out-of-sample predictions on test period
            # -----------------------------------------------------------------

            # Get test period data
            df_test_period = df_full[
                (df_full['Date'] >= fold_config['test_start']) & 
                (df_full['Date'] <= fold_config['test_end'])
            ].copy()


            predictions_dict, actuals_dict, all_preds, all_acts = generate_out_of_sample_predictions_vectorized_exog(
                model=model,
                df_test_period=df_test_period,
                df_full=df_full,
                fold_config=fold_config,
                exog_cols=exog_columns,
                scaler_X=scaler_X,
                scaler_y=scaler_y,
                ts_key_to_idx=ts_key_to_idx,
                n_ts_keys=n_series,
                seq_length=SEQ_LENGTH,
                embargo=EMBARGO,
                device=device
            )


        # -----------------------------------------------------------------
        # Create predictions DataFrame and save as CSV
        # -----------------------------------------------------------------
        predictions_data = []
        for ts_key, preds in predictions_dict.items():
            acts = actuals_dict[ts_key]
            # Get the dates for this time series in the test period
            ts_group = df_test_period[df_test_period['ts_key'] == ts_key].sort_values('Date')
            dates = ts_group['Date'].values[:len(preds)]
            
            for i, (date, pred, actual) in enumerate(zip(dates, preds, acts)):
                predictions_data.append({
                    'ts_key': ts_key,
                    'Date': pd.to_datetime(date),
                    'pred': pred,
                    'true': actual
                })
        
        predictions_df = pd.DataFrame(predictions_data)
        predictions_csv_path = os.path.join(fold_output_dir, 'predictions.csv')
        predictions_df.to_csv(predictions_csv_path, index=False)
        print(f"✓ Saved predictions to: predictions.csv ({len(predictions_df)} rows)")
        
        # -----------------------------------------------------------------
        # Calculate SMAPE distribution
        # -----------------------------------------------------------------
        smape_distribution_df = calculate_smape_distribution(predictions_dict, actuals_dict)
        
        # Calculate category counts and percentages
        category_counts = smape_distribution_df['category'].value_counts()
        total_series = len(smape_distribution_df)
        
        # Define category order
        category_order = ['<10%', '10-20%', '20-30%', '30-40%', '>40%']
        category_distribution = {}
        
        for cat in category_order:
            count = category_counts.get(cat, 0)
            percentage = (count / total_series * 100) if total_series > 0 else 0
            category_distribution[cat] = {
                'count': count,
                'percentage': percentage
            }
        
        fold_smape_distributions.append({
            'fold': fold_config['name'],
            **{f'{cat}_count': category_distribution[cat]['count'] for cat in category_order},
            **{f'{cat}_pct': category_distribution[cat]['percentage'] for cat in category_order}
        })
        
        # Save SMAPE distribution per time series
        smape_distribution_df.to_csv(os.path.join(fold_output_dir, 'smape_per_ts.csv'), index=False)
        
        print(f"\nSMAPE Distribution for {fold_config['name']}:")
        print(f"  Time series evaluated: {total_series}")
        for cat in category_order:
            count = category_distribution[cat]['count']
            pct = category_distribution[cat]['percentage']
            print(f"  {cat:>10}: {count:4d} series ({pct:5.1f}%)")
        
        # -----------------------------------------------------------------
        # STEP 4: Calculate metrics
        # -----------------------------------------------------------------
        print("\n" + "-"*60)
        print("STEP 4: Evaluation Metrics")
        print("-"*60)
        
        mse = mean_squared_error(all_acts, all_preds)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(all_acts, all_preds)
        r2 = r2_score(all_acts, all_preds)
        smape_score = smape(all_acts, all_preds)
        
        fold_metrics.append({
            'fold': fold_config['name'],
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'smape': smape_score
        })
        
        print(f"\n{fold_config['name']} Out-of-Sample Metrics:")
        print(f"  MSE:   {mse:.2f}")
        print(f"  RMSE:  {rmse:.2f}")
        print(f"  MAE:   {mae:.2f}")
        print(f"  R²:    {r2:.4f}")
        print(f"  SMAPE: {smape_score:.2f}%")
        
        # Log final test metrics to W&B
        if wandb_run is not None:
            wandb_run.log({
                "test_mse": mse,
                "test_rmse": rmse,
                "test_mae": mae,
                "test_r2": r2,
                "test_smape": smape_score
            })
            
            # Log SMAPE distribution
            for cat in category_order:
                wandb_run.log({
                    f"smape_dist_{cat}_count": category_distribution[cat]['count'],
                    f"smape_dist_{cat}_pct": category_distribution[cat]['percentage']
                })
        
        # Save model checkpoint (skip for baseline and neuralforecast models)
        if MODEL not in ['BASELINE', 'NBEATS', 'NBEATSx']:
            torch.save({
                'model_state_dict': best_model_state,
                'input_size': INPUT_SIZE,
                'scaler_X': scaler_X,
                'scaler_y': scaler_y,
                'n_series': n_series,  # Updated from n_ts_keys to n_series for vectorized dataset
                'ts_key_to_idx': ts_key_to_idx,
                'seq_length': SEQ_LENGTH,
                'fold_config': fold_config,
                'metrics': fold_metrics[-1]
            }, os.path.join(fold_output_dir, 'model_checkpoint.pth'))
        
        # Visualization (skip for baseline and neuralforecast models)
        if MODEL not in ['BASELINE', 'NBEATS', 'NBEATSx']:
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            
            # Training history
            axes[0].plot(train_losses, label='Training Loss', linewidth=2, color='#2E86AB')
            axes[0].plot(val_losses, label='Validation Loss', linewidth=2, color='#A23B72')
            axes[0].set_xlabel('Epoch', fontsize=12)
            axes[0].set_ylabel('Loss (MAE)', fontsize=12)
            axes[0].set_title(f'{MODEL} - {fold_config["name"]} ({fold_config["test_start"]} to {fold_config["test_end"]}) - Training History', fontsize=13, fontweight='bold')
            axes[0].legend(fontsize=10)
            axes[0].grid(True, alpha=0.3)
            
            # Predictions vs Actuals
            axes[1].scatter(all_acts, all_preds, alpha=0.3, s=10, color='#2E86AB')
            axes[1].plot([all_acts.min(), all_acts.max()], 
                    [all_acts.min(), all_acts.max()], 
                    'r--', linewidth=2, label='Perfect Prediction')
            axes[1].set_xlabel('Actual Values', fontsize=12)
            axes[1].set_ylabel('Predicted Values', fontsize=12)
            axes[1].set_title(f'{MODEL} - {fold_config["name"]} ({fold_config["test_start"]} to {fold_config["test_end"]}) - Predictions (R²={r2:.4f}, SMAPE={smape_score:.2f}%)', 
                        fontsize=13, fontweight='bold')
            axes[1].legend(fontsize=10)
            axes[1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(os.path.join(fold_output_dir, 'fold_results.png'), dpi=300, bbox_inches='tight')
            plt.close()
        
        # Predictions vs Actuals (for all models including BASELINE)
        fig, ax = plt.subplots(figsize=(10, 8))
        ax.scatter(all_acts, all_preds, alpha=0.3, s=10, color='#2E86AB')
        ax.plot([all_acts.min(), all_acts.max()], 
            [all_acts.min(), all_acts.max()], 
            'r--', linewidth=2, label='Perfect Prediction')
        ax.set_xlabel('Actual Values', fontsize=12)
        ax.set_ylabel('Predicted Values', fontsize=12)
        ax.set_title(f'{fold_config["name"]} - Predictions (R²={r2:.4f}, SMAPE={smape_score:.2f}%)', 
                 fontsize=13, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(fold_output_dir, 'predictions_vs_actuals.png'), dpi=300, bbox_inches='tight')
        plt.close()
        
        # Finish W&B run for this fold
        if wandb_run is not None:
            wandb_run.finish()
            print(f"✓ Finished W&B run for {fold_config['name']}")
        
        print(f"\n✓ Saved fold results to: {fold_output_dir}")

    # ========================================================================
    # SMAPE DISTRIBUTION ANALYSIS
    # ========================================================================
    
    print("\n" + "="*80)
    print("SMAPE DISTRIBUTION ANALYSIS")
    print("="*80)
    
    smape_dist_df = pd.DataFrame(fold_smape_distributions)
    
    print("\nPer-Fold SMAPE Distribution:")
    category_order = ['<10%', '10-20%', '20-30%', '30-40%', '>40%']
    
    for _, row in smape_dist_df.iterrows():
        print(f"\n{row['fold']}:")
        for cat in category_order:
            count = row[f'{cat}_count']
            pct = row[f'{cat}_pct']
            print(f"  {cat:>10}: {count:4.0f} series ({pct:5.1f}%)")
    
    print("\n" + "-"*60)
    print("AVERAGE SMAPE DISTRIBUTION ACROSS FOLDS:")
    print("-"*60)
    
    for cat in category_order:
        avg_count = smape_dist_df[f'{cat}_count'].mean()
        avg_pct = smape_dist_df[f'{cat}_pct'].mean()
        std_pct = smape_dist_df[f'{cat}_pct'].std()
        print(f"  {cat:>10}: {avg_pct:5.1f}% ± {std_pct:4.1f}% ({avg_count:.0f} series avg)")
    
    # Save SMAPE distribution summary
    smape_dist_df.to_csv(os.path.join(OUTPUT_PATH, 'smape_distribution.csv'), index=False)
    
    print(f"\n✓ Saved metrics to: {OUTPUT_PATH}/fold_metrics.csv")
    print(f"✓ Saved summary to: {OUTPUT_PATH}/summary_metrics.csv")
    print(f"✓ Saved SMAPE distribution to: {OUTPUT_PATH}/smape_distribution.csv")
    
    # ========================================================================
    # FINAL RESULTS: Average metrics across all folds
    # ========================================================================
    
    print("\n" + "="*80)
    print("FINAL RESULTS: AVERAGE METRICS ACROSS ALL FOLDS")
    print("="*80)
    
    metrics_df = pd.DataFrame(fold_metrics)
    
    print("\nPer-Fold Metrics:")
    print(metrics_df.to_string(index=False))
    
    print("\n" + "-"*60)
    print("AVERAGE PERFORMANCE:")
    print("-"*60)
    avg_metrics = metrics_df[['mse', 'rmse', 'mae', 'r2', 'smape']].mean()
    std_metrics = metrics_df[['mse', 'rmse', 'mae', 'r2', 'smape']].std()
    
    print(f"  MSE:   {avg_metrics['mse']:.2f} ± {std_metrics['mse']:.2f}")
    print(f"  RMSE:  {avg_metrics['rmse']:.2f} ± {std_metrics['rmse']:.2f}")
    print(f"  MAE:   {avg_metrics['mae']:.2f} ± {std_metrics['mae']:.2f}")
    print(f"  R²:    {avg_metrics['r2']:.4f} ± {std_metrics['r2']:.4f}")
    print(f"  SMAPE: {avg_metrics['smape']:.2f}% ± {std_metrics['smape']:.2f}%")
    
    # Save metrics to CSV
    metrics_df.to_csv(os.path.join(OUTPUT_PATH, 'fold_metrics.csv'), index=False)
    
    # Save summary
    summary_dict = {
        'metric': ['MSE', 'RMSE', 'MAE', 'R²', 'SMAPE'],
        'mean': [avg_metrics['mse'], avg_metrics['rmse'], avg_metrics['mae'], 
                 avg_metrics['r2'], avg_metrics['smape']],
        'std': [std_metrics['mse'], std_metrics['rmse'], std_metrics['mae'], 
                std_metrics['r2'], std_metrics['smape']]
    }
    summary_df = pd.DataFrame(summary_dict)
    summary_df.to_csv(os.path.join(OUTPUT_PATH, 'summary_metrics.csv'), index=False)
    
    # ========================================================================
    # SAVE COMPREHENSIVE RESULTS TO TXT FILE
    # ========================================================================
    
    TOTAL_SCRIPT_RUNTIME = (time.time() - SCRIPT_START_TIME) / 60.0  # in minutes
    print(f"\nTotal script runtime: {TOTAL_SCRIPT_RUNTIME:.1f} minutes")

    results_txt_path = os.path.join(OUTPUT_PATH, 'final_results_summary.txt')
    
    with open(results_txt_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write(f"{MODEL} MODEL - FINAL EVALUATION RESULTS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Model: {MODEL}\n")
        if best_trial is not None:
            f.write("Best Hyperparameters:\n")
            for key, value in best_trial.params.items():
                f.write(f"    {key}: {value}\n")
        
        # Total optimization time
        if OPTIMIZE_HYPERPARAMETERS and best_trial is not None:
            f.write(f"\nTotal Hyperparameter Optimization Time: {TOTAL_OPTIMIZATION_TIME:.1f} minutes\n")

        # Training time per fold
        f.write("\nTraining Time per Fold:\n")
        for fold_name, training_time in TOTAL_TRAINING_TIME_FOLDS.items():
            f.write(f"  {fold_name}: {training_time:.1f} seconds\n")
        
        f.write(f"\nTotal Script Runtime: {TOTAL_SCRIPT_RUNTIME:.1f} minutes\n")
        f.write("\n")


        # 1. FINAL RESULTS: AVERAGE METRICS ACROSS ALL FOLDS
        f.write("="*80 + "\n")
        f.write("1. FINAL RESULTS: AVERAGE METRICS ACROSS ALL FOLDS\n")
        f.write("="*80 + "\n\n")
        
        f.write("Per-Fold Metrics:\n")
        f.write(metrics_df.to_string(index=False) + "\n\n")
        
        # 4. AVERAGE PERFORMANCE
        f.write("-"*60 + "\n")
        f.write("4. AVERAGE PERFORMANCE:\n")
        f.write("-"*60 + "\n")
        f.write(f"  MSE:   {avg_metrics['mse']:.2f} ± {std_metrics['mse']:.2f}\n")
        f.write(f"  RMSE:  {avg_metrics['rmse']:.2f} ± {std_metrics['rmse']:.2f}\n")
        f.write(f"  MAE:   {avg_metrics['mae']:.2f} ± {std_metrics['mae']:.2f}\n")
        f.write(f"  R²:    {avg_metrics['r2']:.4f} ± {std_metrics['r2']:.4f}\n")
        f.write(f"  SMAPE: {avg_metrics['smape']:.2f}% ± {std_metrics['smape']:.2f}%\n\n")
        
        # 2. SMAPE DISTRIBUTION ANALYSIS per fold
        f.write("="*80 + "\n")
        f.write("2. SMAPE DISTRIBUTION ANALYSIS PER FOLD\n")
        f.write("="*80 + "\n\n")
        
        for _, row in smape_dist_df.iterrows():
            f.write(f"{row['fold']}:\n")
            for cat in category_order:
                count = row[f'{cat}_count']
                pct = row[f'{cat}_pct']
                f.write(f"  {cat:>10}: {count:4.0f} series ({pct:5.1f}%)\n")
            f.write("\n")
        
        # 3. AVERAGE SMAPE DISTRIBUTION ACROSS FOLDS
        f.write("-"*60 + "\n")
        f.write("3. AVERAGE SMAPE DISTRIBUTION ACROSS FOLDS:\n")
        f.write("-"*60 + "\n")
        
        for cat in category_order:
            avg_count = smape_dist_df[f'{cat}_count'].mean()
            avg_pct = smape_dist_df[f'{cat}_pct'].mean()
            std_pct = smape_dist_df[f'{cat}_pct'].std()
            f.write(f"  {cat:>10}: {avg_pct:5.1f}% ± {std_pct:4.1f}% ({avg_count:.0f} series avg)\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"\n✓ Saved metrics to: {OUTPUT_PATH}/fold_metrics.csv")
    print(f"✓ Saved summary to: {OUTPUT_PATH}/summary_metrics.csv")
    print(f"✓ Saved final results summary to: {OUTPUT_PATH}/final_results_summary.txt")
    print("\n" + "="*80)
    
    
    